# DPO Rewriter Data Generation
Generates `data/dpo/{train,val}.jsonl` — preference pairs `(chosen_rewrite, rejected_rewrite)` per `(question, persona)` used to train the DPO query rewriter.

**Approach (proxy-judge, remote rewriter):** For each `(question, persona)`, sample N=3 rewrites at temperatures 0.2 / 0.5 / 0.9 from a remote Grok model (`grok-4-1-fast`); call a separate LLM judge (`gpt-5.6-luna`, reasoning disabled so its 16-token reply budget is spent on the score rather than on reasoning tokens) once per rewrite to rate how well it would help retrieve the right study material for that learner; pair the highest- and lowest-scoring rewrites as `(chosen, rejected)`. Cross-persona negatives (scholar's best → crammer's rejected) are added for free. All rewriter and judge calls within a question are parallelised with `ThreadPoolExecutor(max_workers=4)`.

Failed rewrite and judge calls — API errors, empty completions, replies that are not a single score in `[0, 1]` — are retried with capped exponential backoff. A candidate whose retry budget is exhausted is dropped rather than scored 0.0, and a persona left with fewer than two candidates is skipped, so no pair is written with `chosen == rejected`.

**Query form:** the rewriter is given the *complete* rendered question — `Passage:` (for grouped reading-comprehension items), `Question:`, `Options:`, `Pairs:`, `Items:` — the same form `gen_ropg_data` used to distil the retriever this rewriter feeds. A bare stem would train the rewriter on an input the pipeline never serves. The gold answer and explanation go to the judge only, as reference context, and never enter the rewriter's prompt or the query. Rows are stamped `format_version: 1`; `rl.dpo_train.load_pairs` refuses anything else, so stem-only pairs cannot be trained on by accident.

> **Why remote rewriter?** An earlier version used a local Gemma-4-E4B via Unsloth. The quality of generated rewrites was insufficient, so the rewriter was replaced with a remote Grok model. The judge remains a separate model family (OpenAI-compatible) to preserve judge independence.

**Kaggle setup checklist**
1. Internet access must be enabled.
2. Add Kaggle secrets: `OPENAI_API_KEY` and `OPENAI_BASE_URL` (judge), `REWRITER_API_KEY` and `REWRITER_BASE_URL` (rewriter/Grok).
3. Attach the `simurgh-data` dataset (contains `questions/` and `splits/`).
4. No GPU required — both models are API calls.

**Colab setup checklist**
1. Set `RUNTIME = "colab"` in the Config cell below.
2. Upload `simurgh-data/` to Google Drive at `MyDrive/simurgh-data/` — must contain `questions/`, `splits/`.
3. Add secrets via Colab Secrets (left sidebar → key icon): `OPENAI_API_KEY`, `OPENAI_BASE_URL`, `REWRITER_API_KEY`, `REWRITER_BASE_URL`.
4. Output is saved to `MyDrive/simurgh-data/dpo/`.

In [ ]:
!pip install -q openai tqdm

## Config

In [ ]:
import os

# ── Runtime selector ─────────────────────────────────────────────────────────
# Set RUNTIME to match where you are running this notebook.
RUNTIME = "kaggle"  # "kaggle" | "colab" | "local"
GDRIVE_BASE = "/content/drive/MyDrive/simurgh-data"  # Colab only

# ── Secrets ───────────────────────────────────────────────────────────────────
if RUNTIME == "kaggle":
    from kaggle_secrets import UserSecretsClient

    _s = UserSecretsClient()
    OPENAI_API_KEY = _s.get_secret("OPENAI_API_KEY")
    OPENAI_BASE_URL = _s.get_secret("OPENAI_BASE_URL")
    REWRITER_API_KEY = _s.get_secret("REWRITER_API_KEY")
    REWRITER_BASE_URL = _s.get_secret("REWRITER_BASE_URL")
elif RUNTIME == "colab":
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    OPENAI_BASE_URL = userdata.get("OPENAI_BASE_URL")
    REWRITER_API_KEY = userdata.get("REWRITER_API_KEY")
    REWRITER_BASE_URL = userdata.get("REWRITER_BASE_URL")
else:  # local
    OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
    OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")
    REWRITER_API_KEY = os.environ.get("REWRITER_API_KEY", "")
    REWRITER_BASE_URL = os.environ.get("REWRITER_BASE_URL")

# ── Paths ─────────────────────────────────────────────────────────────────────
if RUNTIME == "kaggle":
    DATASET_SLUG = "simurgh-data"
    DATA_ROOT = f"/kaggle/input/datasets/alirezahsn/{DATASET_SLUG}"
    OUTPUT_DIR = "/kaggle/working/data/dpo"
elif RUNTIME == "colab":
    DATA_ROOT = GDRIVE_BASE
    OUTPUT_DIR = f"{GDRIVE_BASE}/dpo"
else:  # local
    DATA_ROOT = "data"
    OUTPUT_DIR = "data/dpo"

# ── Inline config (mirrors configs/datagen_dpo.yaml) ─────────────────────────
CFG = {
    "rewriter": {
        "model": "grok-4-1-fast",
        "temperatures": [0.2, 0.5, 0.9],
        "max_completion_tokens": 300,
        "max_workers": 4,
        "max_attempts": 3,
        "initial_backoff_seconds": 1.0,
        "backoff_multiplier": 2.0,
        "max_backoff_seconds": 8.0,
    },
    "judge": {
        "model": "gpt-5.6-luna",
        # Luna is a reasoning model: with reasoning enabled, reasoning tokens consume the
        # 16-token completion budget and the response comes back empty.
        "reasoning_effort": "none",
        "temperature": 1.0,
        "max_completion_tokens": 16,
        "max_attempts": 3,
        "initial_backoff_seconds": 1.0,
        "backoff_multiplier": 2.0,
        "max_backoff_seconds": 8.0,
    },
    "cross_persona_threshold": 0.25,
    "data": {
        "questions_dir": f"{DATA_ROOT}/questions",
        "splits_dir": f"{DATA_ROOT}/splits",
        "output_dir": OUTPUT_DIR,
    },
    "seed": 42,
}

## Core classes

In [ ]:
import openai


class OpenAICompatClient:
    def __init__(
        self,
        base_url,
        api_key,
        model,
        temperature=0.0,
        max_completion_tokens=16,
        reasoning_effort=None,
    ):
        self.client = openai.OpenAI(base_url=base_url, api_key=api_key)
        self.model = model
        self.temperature = temperature
        self.max_completion_tokens = max_completion_tokens
        self.reasoning_effort = reasoning_effort

    def chat(self, messages: list) -> str:
        kwargs = {
            "model": self.model,
            "messages": messages,
            "temperature": self.temperature,
            "max_completion_tokens": self.max_completion_tokens,
        }
        if self.reasoning_effort is not None:
            kwargs["reasoning_effort"] = self.reasoning_effort
        resp = self.client.chat.completions.create(**kwargs)
        return resp.choices[0].message.content or ""

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Profile:
    id: str
    split: str
    rendered: str


PERSONAS = {
    "crammer": Profile(
        id="crammer",
        split="train",
        rendered=(
            "A ninth-grader who finds the textbook hard to follow and has little background "
            "on this topic. Mainly wants to pass the exam — give the answer and what is needed "
            "to score — but it must be spelled out simply, step by step, with examples."
        ),
    ),
    "scholar": Profile(
        id="scholar",
        split="train",
        rendered=(
            "A ninth-grader who reads dense material easily and has solid background on this "
            "topic. Wants to understand the underlying why and how, and the connections between "
            "ideas. Prefers a terse, high-level treatment without hand-holding or padding."
        ),
    ),
    "steady": Profile(
        id="steady",
        split="train",
        rendered=(
            "A capable ninth-grader with average background on this topic. Wants a correct "
            "answer with a brief justification, balanced toward exam needs. Does not need "
            "elaborate scaffolding, but does appreciate a one-line reason."
        ),
    ),
}


def render_profile(persona_id: str) -> str:
    return PERSONAS[persona_id].rendered


def train_personas() -> list:
    return [p for p in PERSONAS.values() if p.split == "train"]

In [ ]:
_REWRITE_SYSTEM = (
    "You are a query rewriting assistant for a Persian educational RAG system. "
    "Given a learner profile and an original question, rewrite the question as a "
    "retrieval query that will surface the most pedagogically useful passages for "
    "that specific learner. "
    "Rules: output ONLY the rewritten query — no explanation, no preamble, no quotes. "
    "Keep it in Persian if the original is Persian. "
    "You may expand abbreviations, add prerequisite terms, or rephrase for clarity, "
    "but do not invent facts or change the question's intent."
)


def build_rewrite_prompt(profile_rendered: str, query: str) -> list:
    user = (
        f"Learner profile: {profile_rendered}\n\n"
        f"Original question: {query}\n\n"
        "Rewritten retrieval query:"
    )
    return [
        {"role": "system", "content": _REWRITE_SYSTEM},
        {"role": "user", "content": user},
    ]

## Helper functions

In [ ]:
import json
import logging
import math
import re
import time
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from typing import TypeVar

from tqdm.auto import tqdm

logging.basicConfig(
    force=True,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# A judge reply must be exactly one score in [0, 1] — nothing looser, so that a truncated
# or chatty response retries instead of silently becoming a real-looking score.
SCORE_RE = re.compile(r"(?:0(?:\.\d+)?|1(?:\.0+)?)")

T = TypeVar("T")


@dataclass(frozen=True)
class RetryPolicy:
    """Exponential, capped backoff budget for one API call site."""

    max_attempts: int = 3
    initial_backoff_seconds: float = 1.0
    backoff_multiplier: float = 2.0
    max_backoff_seconds: float = 8.0

    @classmethod
    def from_config(cls, block: dict, block_name: str) -> "RetryPolicy":
        policy = cls(
            max_attempts=block.get("max_attempts", 3),
            initial_backoff_seconds=block.get("initial_backoff_seconds", 1.0),
            backoff_multiplier=block.get("backoff_multiplier", 2.0),
            max_backoff_seconds=block.get("max_backoff_seconds", 8.0),
        )
        policy.validate(block_name)
        return policy

    def validate(self, block_name: str) -> None:
        if (
            isinstance(self.max_attempts, bool)
            or not isinstance(self.max_attempts, int)
            or self.max_attempts < 1
        ):
            raise ValueError(f"{block_name}.max_attempts must be an integer >= 1")
        for field_name, value in (
            ("initial_backoff_seconds", self.initial_backoff_seconds),
            ("backoff_multiplier", self.backoff_multiplier),
            ("max_backoff_seconds", self.max_backoff_seconds),
        ):
            if (
                isinstance(value, bool)
                or not isinstance(value, (int, float))
                or not math.isfinite(value)
            ):
                raise ValueError(f"{block_name}.{field_name} must be a finite number")
        if self.initial_backoff_seconds < 0:
            raise ValueError(f"{block_name}.initial_backoff_seconds must be >= 0")
        if self.backoff_multiplier < 1:
            raise ValueError(f"{block_name}.backoff_multiplier must be >= 1")
        if self.max_backoff_seconds < self.initial_backoff_seconds:
            raise ValueError(
                f"{block_name}.max_backoff_seconds must be >= {block_name}.initial_backoff_seconds"
            )


class CandidateError(RuntimeError):
    """Raised when a rewrite or its score cannot be produced within the retry budget."""


def _call_with_retry(label: str, call: Callable[[], T], policy: RetryPolicy) -> T:
    backoff_seconds = policy.initial_backoff_seconds
    for attempt in range(1, policy.max_attempts + 1):
        try:
            return call()
        except Exception as exc:
            logger.warning(
                "%s failed (attempt %d/%d)", label, attempt, policy.max_attempts, exc_info=True
            )
            if attempt == policy.max_attempts:
                raise CandidateError(
                    f"{label} failed after {policy.max_attempts} attempts"
                ) from exc
            time.sleep(backoff_seconds)
            backoff_seconds = min(
                backoff_seconds * policy.backoff_multiplier, policy.max_backoff_seconds
            )
    raise CandidateError(f"{label} failed")


DPO_JUDGE_SYSTEM = (
    "You are an expert Persian language tutor evaluating query rewrites "
    "for a RAG retrieval system."
)

# Bumped whenever the meaning of a written row changes. Version 1 is the first format
# whose `query` is the complete rendered question rather than the bare stem, and whose
# scores come from a judge that saw the gold answer. rl.dpo_train.load_pairs refuses any
# other version, so stem-only pairs cannot be trained on by accident.
DPO_OUTPUT_FORMAT_VERSION = 1


def build_judge_messages(
    persona_rendered: str,
    original_query: str,
    rewrite: str,
    answer=None,
    explanation=None,
) -> list:
    # Gold fields are judge context only. They tell the judge which material actually
    # resolves the question, and they never reach the rewriter — a rewrite conditioned on
    # the answer would leak it into the retrieval query.
    reference_sections = []
    if answer is not None:
        rendered_answer = answer if isinstance(answer, str) else render_question_value(answer)
        reference_sections.append(f"Gold answer/reference:\n{rendered_answer}")
    if explanation is not None:
        reference_sections.append(f"Gold explanation/rubric:\n{explanation}")
    reference_context = ""
    if reference_sections:
        reference_context = (
            "Reference answer and rubric (judge context only; the rewriter never saw this):\n"
            + "\n\n".join(reference_sections)
            + "\n\n"
        )

    user = (
        "A student with the following profile is searching for study material:\n"
        f"Profile: {persona_rendered}\n\n"
        f"Original exam question: {original_query}\n\n"
        f"{reference_context}"
        f"Candidate rewrite: {rewrite}\n\n"
        "Rate 0.0\u20131.0 how well this rewrite would help retrieve the right study material "
        "for this specific student. A good rewrite should:\n"
        "  - Preserve the original question's meaning\n"
        "  - Use vocabulary and framing that matches the student's profile\n"
        "  - Be specific enough to surface relevant passages at the right depth\n\n"
        "Respond with a single decimal number only, e.g. 0.61"
    )
    return [
        {"role": "system", "content": DPO_JUDGE_SYSTEM},
        {"role": "user", "content": user},
    ]


def parse_score(response: str) -> float:
    """Parse a judge response containing exactly one score in [0, 1]."""
    if not isinstance(response, str):
        raise ValueError(f"Judge response is not a score string: {response!r}")
    stripped = response.strip()
    if SCORE_RE.fullmatch(stripped) is None:
        raise ValueError(f"Judge response is not a single score in [0, 1]: {response!r}")
    score = float(stripped)
    if not math.isfinite(score) or not 0.0 <= score <= 1.0:
        raise ValueError(f"Judge response is not a finite score in [0, 1]: {response!r}")
    return score


def count_question_files(questions_dir: Path) -> int:
    count = sum(1 for _ in questions_dir.glob("*.json"))
    logger.info("Found %d question files in %s", count, questions_dir)
    return count


# ── Question rendering ────────────────────────────────────────────────────────
# The retriever this rewriter feeds was distilled on the query form produced here
# (Passage: / Question: / Options: / Pairs: / Items: sections), so a bare stem would
# query the encoder with a form it never saw in training. Kept identical to
# src/data/questions.py and to the same block in gen_ropg_data.ipynb.


@dataclass(frozen=True)
class QuestionContext:
    query: str
    answer: object | None = None
    explanation: str | None = None


def render_question_value(value) -> str:
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def render_numbered_section(label: str, values) -> str | None:
    if values is None:
        return None
    entries = values if isinstance(values, list) else [values]
    if not entries:
        return None
    rendered = "\n".join(
        f"{index}. {render_question_value(value)}" for index, value in enumerate(entries, start=1)
    )
    return f"{label}:\n{rendered}"


def load_question(exam_stem: str, qid: str, questions_dir: Path) -> QuestionContext:
    qfile = questions_dir / f"{exam_stem}.json"
    if not qfile.exists():
        raise FileNotFoundError(f"Question file not found: {qfile}")
    data = json.loads(qfile.read_text(encoding="utf-8"))
    for q in data.get("questions", []):
        if q["id"] == qid:
            sections = []
            group_id = q.get("group_id")
            if group_id is not None:
                passages = data.get("passages", data.get("passage", []))
                matching_passage = None
                if isinstance(passages, dict):
                    if passages.get("id") == group_id or passages.get("group_id") == group_id:
                        matching_passage = passages
                    else:
                        matching_passage = passages.get(group_id)
                        if matching_passage is None:
                            matching_passage = passages.get(str(group_id))
                elif isinstance(passages, list):
                    for passage in passages:
                        if not isinstance(passage, dict):
                            continue
                        if passage.get("id") == group_id or passage.get("group_id") == group_id:
                            matching_passage = passage
                            break
                if matching_passage is None:
                    raise KeyError(
                        f"Question {qid!r} in {qfile} references group_id={group_id!r}, "
                        "but no matching top-level passage exists"
                    )
                if isinstance(matching_passage, dict):
                    passage_text = matching_passage.get("text", matching_passage.get("passage"))
                else:
                    passage_text = matching_passage
                if passage_text is None:
                    raise KeyError(
                        f"Question {qid!r} in {qfile} references group_id={group_id!r}, "
                        "but the matching top-level passage has no text"
                    )
                sections.append(f"Passage:\n{render_question_value(passage_text)}")

            sections.append(f"Question:\n{render_question_value(q['stem'])}")

            options_section = render_numbered_section("Options", q.get("options"))
            if options_section is not None:
                sections.append(options_section)

            pairs = q.get("pairs")
            if isinstance(pairs, dict):
                for side in ("left", "right"):
                    pair_section = render_numbered_section(f"Pairs ({side})", pairs.get(side))
                    if pair_section is not None:
                        sections.append(pair_section)
            elif pairs is not None:
                pair_section = render_numbered_section("Pairs", pairs)
                if pair_section is not None:
                    sections.append(pair_section)

            items_section = render_numbered_section("Items", q.get("items"))
            if items_section is not None:
                sections.append(items_section)

            return QuestionContext(
                query="\n\n".join(sections),
                answer=q.get("answer"),
                explanation=q.get("explanation"),
            )
    raise KeyError(f"Question {qid!r} not found in {qfile}")


def serialize_record(
    question_ref: str, query: str, persona_id: str, chosen: str, rejected: str
) -> str:
    record = {
        "format_version": DPO_OUTPUT_FORMAT_VERSION,
        "question_ref": question_ref,
        "persona_id": persona_id,
        "query": query,
        "chosen": chosen,
        "rejected": rejected,
    }
    return json.dumps(record, ensure_ascii=False) + "\n"


def read_split_qids(splits_dir: Path, split_name: str) -> list:
    split_file = splits_dir / f"{split_name}_qids.txt"
    if not split_file.exists():
        raise FileNotFoundError(f"Split file not found: {split_file}")
    pairs = []
    for line in split_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        exam_stem, qid = line.split(":", 1)
        pairs.append((exam_stem.strip(), qid.strip()))
    return pairs

## Run pipeline

In [ ]:
# Health check: 3 rewrites + judge scores for one sample question.
# Runs before the full pipeline to confirm both models are working. The sample uses the
# same rendered form the pipeline builds (a `Question:` section plus a gold answer that
# only the judge sees), so this also exercises the reference-context path.

SAMPLE_QUESTION = QuestionContext(
    query=(
        "Question:\n"
        "انرژی جنبشی یک جسم با جرم ۲ کیلوگرم که با سرعت ۳ متر بر ثانیه حرکت می‌کند چقدر است؟"
    ),
    answer="۹ ژول",
)
CHECK_PERSONA = "crammer"

check_profile = PERSONAS[CHECK_PERSONA].rendered
_check_judge = OpenAICompatClient(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    model=CFG["judge"]["model"],
    temperature=CFG["judge"]["temperature"],
    max_completion_tokens=CFG["judge"]["max_completion_tokens"],
    reasoning_effort=CFG["judge"].get("reasoning_effort"),
)

print(f"Persona : {CHECK_PERSONA}")
print(f"Query   : {SAMPLE_QUESTION.query}\n")

for temp in CFG["rewriter"]["temperatures"]:
    _rw_client = OpenAICompatClient(
        base_url=REWRITER_BASE_URL,
        api_key=REWRITER_API_KEY,
        model=CFG["rewriter"]["model"],
        temperature=temp,
        max_completion_tokens=CFG["rewriter"]["max_completion_tokens"],
    )
    msgs = build_rewrite_prompt(check_profile, SAMPLE_QUESTION.query)
    rewrite = _rw_client.chat(msgs)
    judge_msgs = build_judge_messages(
        check_profile,
        SAMPLE_QUESTION.query,
        rewrite,
        answer=SAMPLE_QUESTION.answer,
        explanation=SAMPLE_QUESTION.explanation,
    )
    raw = _check_judge.chat(judge_msgs)
    score = parse_score(raw)
    print(f"  temp={temp:.1f}  score={score:.2f}  {rewrite[:100]}")

print("\nHealth check complete.")


In [ ]:
import random
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

from tqdm.auto import tqdm

random.seed(CFG["seed"])

questions_dir = Path(CFG["data"]["questions_dir"])
splits_dir = Path(CFG["data"]["splits_dir"])
output_dir = Path(CFG["data"]["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

temperatures = CFG["rewriter"]["temperatures"]
max_workers = CFG["rewriter"]["max_workers"]
cross_threshold = CFG["cross_persona_threshold"]
train_profiles = train_personas()
persona_ids = [p.id for p in train_profiles]

count_question_files(questions_dir)

# One OpenAICompatClient per temperature (same Grok model, different temp).
rewriter_clients = {
    temp: OpenAICompatClient(
        base_url=REWRITER_BASE_URL,
        api_key=REWRITER_API_KEY,
        model=CFG["rewriter"]["model"],
        temperature=temp,
        max_completion_tokens=CFG["rewriter"]["max_completion_tokens"],
    )
    for temp in temperatures
}

judge = OpenAICompatClient(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    model=CFG["judge"]["model"],
    temperature=CFG["judge"]["temperature"],
    max_completion_tokens=CFG["judge"]["max_completion_tokens"],
    reasoning_effort=CFG["judge"].get("reasoning_effort"),
)

rewrite_policy = RetryPolicy.from_config(CFG["rewriter"], "rewriter")
judge_policy = RetryPolicy.from_config(CFG["judge"], "judge")

logger.info(
    "Rewriter: %s temps=%s max_completion_tokens=%d max_workers=%d attempts=%d",
    CFG["rewriter"]["model"],
    temperatures,
    CFG["rewriter"]["max_completion_tokens"],
    max_workers,
    rewrite_policy.max_attempts,
)
logger.info(
    "Judge: %s reasoning_effort=%s temperature=%s max_completion_tokens=%d attempts=%d",
    CFG["judge"]["model"],
    CFG["judge"].get("reasoning_effort"),
    CFG["judge"]["temperature"],
    CFG["judge"]["max_completion_tokens"],
    judge_policy.max_attempts,
)


def _run_one_combo(
    persona_id,
    temp,
    rw_client,
    judge_client,
    persona_rendered,
    question,
    rewrite_policy,
    judge_policy,
):
    label = f"persona={persona_id} temp={temp:.1f}"

    def _rewrite_once():
        rewrite = rw_client.chat(build_rewrite_prompt(persona_rendered, question.query)).strip()
        if not rewrite:
            raise ValueError("Rewriter returned an empty completion")
        return rewrite

    def _score_once():
        judge_msgs = build_judge_messages(
            persona_rendered,
            question.query,
            rewrite,
            answer=question.answer,
            explanation=question.explanation,
        )
        return parse_score(judge_client.chat(judge_msgs))

    try:
        rewrite = _call_with_retry(f"Rewrite {label}", _rewrite_once, rewrite_policy)
    except CandidateError:
        logger.error("Dropping candidate — rewrite exhausted its retry budget: %s", label)
        return persona_id, None, 0.0

    try:
        score = _call_with_retry(f"Judge {label}", _score_once, judge_policy)
    except CandidateError:
        logger.error("Dropping candidate — judge exhausted its retry budget: %s", label)
        return persona_id, None, 0.0

    return persona_id, rewrite, score


for split_name in ("train", "val"):
    try:
        qid_pairs = read_split_qids(splits_dir, split_name)
    except FileNotFoundError as e:
        logger.warning("%s — skipping %s split", e, split_name)
        continue

    output_path = output_dir / f"{split_name}.jsonl"
    records_written = 0
    logger.info(
        "Processing %s split (%d questions) → %s", split_name, len(qid_pairs), output_path
    )

    with output_path.open("w", encoding="utf-8") as fh:
        for exam_stem, qid in tqdm(qid_pairs, desc=split_name, unit="q"):
            try:
                question = load_question(exam_stem, qid, questions_dir)
            except (FileNotFoundError, KeyError, json.JSONDecodeError) as e:
                logger.warning("Skipping %s:%s: %s", exam_stem, qid, e)
                continue

            question_ref = f"{exam_stem}:{qid}"

            combos = [(pid, temp) for pid in persona_ids for temp in temperatures]
            candidates_by_persona = defaultdict(list)

            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                future_map = {
                    executor.submit(
                        _run_one_combo,
                        pid, temp,
                        rewriter_clients[temp],
                        judge,
                        render_profile(pid),
                        question,
                        rewrite_policy,
                        judge_policy,
                    ): (pid, temp)
                    for pid, temp in combos
                }
                for future in as_completed(future_map):
                    pid, rewrite, score = future.result()
                    if rewrite is not None:
                        candidates_by_persona[pid].append((rewrite, score))

            best_for_persona = {}
            best_score_for_persona = {}

            for persona_id in persona_ids:
                candidates = candidates_by_persona[persona_id]
                if len(candidates) < 2:
                    logger.error(
                        "Only %d candidate(s) for %s:%s persona=%s — skipping persona",
                        len(candidates),
                        exam_stem,
                        qid,
                        persona_id,
                    )
                    continue
                candidates.sort(key=lambda x: x[1], reverse=True)
                chosen, chosen_score = candidates[0]
                rejected, _ = candidates[-1]
                best_for_persona[persona_id] = chosen
                best_score_for_persona[persona_id] = chosen_score
                fh.write(serialize_record(question_ref, question.query, persona_id, chosen, rejected))
                records_written += 1

            for i, pid_a in enumerate(persona_ids):
                if pid_a not in best_for_persona:
                    continue
                for pid_b in persona_ids[i + 1:]:
                    if pid_b not in best_for_persona:
                        continue
                    if abs(best_score_for_persona[pid_a] - best_score_for_persona[pid_b]) <= cross_threshold:
                        continue
                    fh.write(serialize_record(question_ref, question.query, pid_a, best_for_persona[pid_a], best_for_persona[pid_b]))
                    records_written += 1
                    fh.write(serialize_record(question_ref, question.query, pid_b, best_for_persona[pid_b], best_for_persona[pid_a]))
                    records_written += 1

    logger.info("Wrote %d records to %s", records_written, output_path)
